# 🚛 Logistics Route Optimization System — Technical Documentation

**Version:** 2.0.0  
**Organization:** AIQ Space Ventures Private Limited  
**Date:** October 2026  

---

This notebook serves as the **comprehensive technical documentation** for the Logistics Route Optimization System. It covers:

1. System Architecture & Technology Stack
2. Pipeline Stages (End-to-End Flow)
3. Valhalla Routing Engine Integration
4. OR-Tools Optimization Engine Integration
5. Constraint System & Penalty Architecture
6. Data Models & API Reference
7. Code Walkthrough (Key Modules)
8. Performance Benchmarks
9. Frontend Dashboard Architecture

---
## 1. System Overview

### 1.1 Problem Statement

A warehouse manager needs to deliver parcels to **N customers** using a fleet of **M vehicles**. Each customer has:
- A delivery **address** (geocoded to lat/lon)
- A **parcel weight** (kg)
- A **delivery time window** (e.g., 09:00 – 17:00)
- A **service time** (time spent at the stop)

Each vehicle has:
- A **capacity limit** (kg)
- A **driver shift** (start time – end time)
- A **cost per km** (operational cost)

**Goal:** Minimize total cost (distance + penalties) while satisfying all constraints.

### 1.2 Why Two Engines?

| Engine | Role | What It Does | What It CANNOT Do |
|--------|------|-------------|-------------------|
| **Valhalla** | Routing Engine | Computes road distances, travel times, GeoJSON geometries | Multi-vehicle assignment, capacity constraints, time windows |
| **OR-Tools** | Optimization Engine | Solves CVRPTW: assigns orders to vehicles, sequences stops | Road-network distance computation, turn-by-turn navigation |

**They work together:**
```
Valhalla (ROAD COST) → OR-Tools (OPTIMAL ASSIGNMENT) → Valhalla (ROAD GEOMETRY)
```

---
## 2. Technology Stack

| Layer | Technology | Version | Purpose |
|-------|-----------|---------|----------|
| Backend Framework | FastAPI | 0.115 | Async REST API with OpenAPI docs |
| Routing Engine | Valhalla | 3.4 | Road-network distance matrix + route geometry |
| Optimization | Google OR-Tools | 9.11 | CVRPTW fleet optimization solver |
| Database | SQLite + aiosqlite | — | Async job/order/route persistence |
| ORM | SQLAlchemy | 2.0 (async) | Database models and queries |
| Geocoding | Nominatim / Ola Maps | — | Address → coordinates resolution |
| Data Parsing | Pandas + openpyxl | 2.2 | Excel/CSV manifest ingestion |
| HTTP Client | httpx | 0.28 | Async Valhalla API calls |
| Frontend | React + TypeScript | 19 | Interactive dashboard |
| Map | Leaflet | 1.9 | Route visualization with dark CARTO tiles |
| Bundler | Vite | 6 | Frontend build system |
| Styling | Tailwind CSS | 3.4 | Utility-first responsive design |

---
## 3. System Architecture Diagram

```
┌─────────────────────────────────────────────────────────────────────────┐
│                      FRONTEND (React + Vite + Leaflet)                  │
│                                                                         │
│  ┌─────────┐  ┌──────────────┐  ┌───────────┐  ┌────────────────────┐  │
│  │ Upload  │  │ Job Timeline │  │  Summary  │  │  Interactive Map   │  │
│  │  Modal  │  │   (Polling)  │  │   Stats   │  │  (Route Polylines) │  │
│  └────┬────┘  └──────┬───────┘  └─────┬─────┘  └────────┬───────────┘  │
│       └──────────────┴────────────────┴──────────────────┘              │
│                              ↕ Axios HTTP                               │
└─────────────────────────────────────────────────────────────────────────┘
                               │
                          /api/v1/*
                               │
┌─────────────────────────────────────────────────────────────────────────┐
│                     BACKEND (FastAPI + Async Python)                     │
│                                                                         │
│  ┌──────────────────────────────────────────────────────────────────┐   │
│  │                  PipelineOrchestrator (pipeline.py)               │   │
│  │                                                                  │   │
│  │  STAGE 1        STAGE 2        STAGE 3       STAGE 4   STAGE 5  │   │
│  │  ┌────────┐    ┌──────────┐   ┌─────────┐  ┌───────┐  ┌──────┐ │   │
│  │  │Manifest│ →  │Geocoding │ → │Valhalla │→ │OR-Tools│→ │Route │ │   │
│  │  │Parser  │    │Service   │   │ Matrix  │  │ Solver │  │Geom  │ │   │
│  │  └────────┘    └──────────┘   └─────────┘  └───────┘  └──────┘ │   │
│  └──────────────────────────────────────────────────────────────────┘   │
│                                                                         │
│  ┌─────────────┐  ┌──────────────┐  ┌───────────────┐                  │
│  │ SQLite DB   │  │ Geocode Cache│  │ Fleet Config  │                  │
│  │ (aiosqlite) │  │  (JSON file) │  │ (DB seeded)   │                  │
│  └─────────────┘  └──────────────┘  └───────────────┘                  │
└─────────────────────────────────────────────────────────────────────────┘
                               │
                    ┌──────────┴──────────┐
                    │  Valhalla Server    │
                    │  (localhost:8002)   │
                    │  OpenStreetMap Data │
                    └─────────────────────┘
```

---
## 4. Pipeline Stages — Detailed Walkthrough

The `PipelineOrchestrator` processes each job through 6 sequential stages:

```
VALIDATING (0-10%) → GEOCODING (10-35%) → MATRIX_GENERATION (35-50%)
    → OPTIMIZING (50-70%) → ROUTE_GENERATION (70-85%) → COMPLETED (100%)
```

### Stage 1: Manifest Validation (VALIDATING)

**Engine:** Pandas  
**Input:** Uploaded Excel (.xlsx/.xls) or CSV file  

**Process:**
1. Parse the file using `pd.read_excel()` or `pd.read_csv()`
2. Validate required columns exist: `consignment_id`, `address`, `weight_kg`
3. Apply defaults for optional fields:
   - `service_time_min` → 5 minutes
   - `window_start` → 08:00 (480 minutes)
   - `window_end` → 20:00 (1200 minutes)
4. Create `Order` records in the database

**Output:** N validated `Order` rows persisted to SQLite

In [ ]:
# Stage 1 Code Reference: ManifestParser (excel_parser.py)
# This is the core validation logic used by the pipeline

import pandas as pd

# Column mapping — flexible header matching
COLUMN_ALIASES = {
    'consignment_id': ['consignment_id', 'order_id', 'id', 'consignment'],
    'address': ['address', 'delivery_address', 'customer_address', 'location'],
    'weight_kg': ['weight_kg', 'weight', 'parcel_weight', 'weight_kgs'],
    'service_time_min': ['service_time_min', 'service_time', 'stop_time'],
    'window_start': ['window_start', 'delivery_start', 'start_time'],
    'window_end': ['window_end', 'delivery_end', 'end_time']
}

# Default values for optional columns
DEFAULTS = {
    'service_time_min': 5,       # 5 minutes per stop
    'window_start': 480,         # 08:00
    'window_end': 1200           # 20:00
}

print("✅ Manifest parser configured with flexible column aliases")
print(f"   Required columns: consignment_id, address, weight_kg")
print(f"   Optional columns: service_time_min, window_start, window_end")

### Stage 2: Geocoding (GEOCODING)

**Engine:** Nominatim (OpenStreetMap) or Ola Maps  
**Input:** Raw address strings from orders  

**Strategy:** Cache-first with persistent JSON file

```
Address → Check geocode_cache.json → HIT? Return cached (lat, lon)
                                   → MISS? Call Nominatim API
                                            → Cache result
                                            → Return (lat, lon)
```

**Key Design Decisions:**
- **Rate limiting:** 1 request/second for Nominatim (their usage policy)
- **Persistent cache:** `geocode_cache.json` survives server restarts
- **Provider factory:** Pluggable providers (nominatim, ola_maps, mock)

In [ ]:
# Stage 2 Code Reference: Geocoding Architecture

geocoding_architecture = """
geocoding/
├── base.py          # Abstract BaseGeocoder interface
│                    # async def geocode(address: str) -> (lat, lon)
│
├── cache.py         # JSON file-based persistent cache
│                    # Wraps any provider with cache-first lookup
│
├── factory.py       # Provider factory pattern
│                    # get_geocoder(provider_name) -> BaseGeocoder
│
├── nominatim.py     # OpenStreetMap Nominatim provider
│                    # Free, rate-limited to 1 req/sec
│
└── ola_maps.py      # Ola Maps geocoding provider
                     # India-optimized, requires API key
"""

print(geocoding_architecture)

# Example cache entry structure
cache_example = {
    "Andheri West, Mumbai, Maharashtra": {
        "latitude": 19.1364,
        "longitude": 72.8296,
        "resolved_address": "Andheri West, Mumbai, Maharashtra, India",
        "provider": "nominatim"
    }
}
print(f"Cache entry example: {cache_example}")

### Stage 3: Valhalla Distance Matrix (MATRIX_GENERATION)

**Engine:** Valhalla `/sources_to_targets` API  
**Input:** N+1 geocoded coordinates (depot + N delivery locations)  

This is where Valhalla computes the **road-network travel costs** between every pair of locations. Unlike Euclidean (straight-line) distance, this accounts for:
- Actual road segments from OpenStreetMap
- One-way streets, highway ramps, turn restrictions
- Road hierarchy (highways vs. local roads)
- Costing model (auto, truck, motorcycle)

**API Call:**
```json
POST http://localhost:8002/sources_to_targets
{
    "sources": [{"lat": 19.055, "lon": 72.875}, ...],
    "targets": [{"lat": 19.055, "lon": 72.875}, ...],
    "costing": "auto"
}
```

**Output:** Two N×N matrices:
- `time_matrix_seconds[i][j]` — Road travel time from i to j (seconds)
- `dist_matrix_meters[i][j]` — Road distance from i to j (meters)

**Latency:** ~50–200ms for 20 locations (compared to multi-second PostgreSQL pgRouting)

In [ ]:
# Stage 3 Code Reference: Valhalla Matrix Generation

# This is the core Valhalla client logic (simplified for documentation)

import json

# Example: Valhalla /sources_to_targets request
valhalla_matrix_request = {
    "sources": [
        {"lat": 19.0550, "lon": 72.8750},   # Depot
        {"lat": 19.1364, "lon": 72.8296},   # Order 1: Andheri
        {"lat": 19.0176, "lon": 72.8562},   # Order 2: Dadar
        {"lat": 19.2183, "lon": 72.9781},   # Order 3: Thane
    ],
    "targets": [
        {"lat": 19.0550, "lon": 72.8750},   # Depot
        {"lat": 19.1364, "lon": 72.8296},   # Order 1
        {"lat": 19.0176, "lon": 72.8562},   # Order 2
        {"lat": 19.2183, "lon": 72.9781},   # Order 3
    ],
    "costing": "auto"
}

print("Valhalla /sources_to_targets Request:")
print(json.dumps(valhalla_matrix_request, indent=2))

# Example response (simplified)
print("\n--- Example Response ---")
print("time_matrix_seconds (4×4):")
example_time = [
    [0,    1920, 1080, 3600],  # From Depot
    [1800, 0,    2400, 2700],  # From Andheri
    [1200, 2520, 0,    4200],  # From Dadar
    [3480, 2580, 4080, 0   ],  # From Thane
]
for row in example_time:
    print(f"  {row}")

print("\ndist_matrix_meters (4×4):")
example_dist = [
    [0,     12400, 6800,  25600],
    [11800, 0,     15200, 18900],
    [7200,  16000, 0,     28400],
    [24800, 17600, 27200, 0    ],
]
for row in example_dist:
    print(f"  {row}")

### Stage 4: OR-Tools CVRPTW Optimization (OPTIMIZING)

**Engine:** Google OR-Tools `RoutingModel`  
**Input:** Distance/time matrices from Valhalla + fleet configuration + constraints  

This is the **core intelligence** of the system. OR-Tools solves the **Capacitated Vehicle Routing Problem with Time Windows (CVRPTW):**

- **Given:** N customers, M vehicles, distance matrix, time matrix
- **Find:** Assignment of customers to vehicles + optimal visit sequence
- **Minimize:** Total cost = travel_cost + penalty_points
- **Subject to:** Capacity, time windows, driver shifts, overtime limits

#### OR-Tools Solver Setup Steps:

```
1. Create RoutingIndexManager(N+1 locations, M vehicles, depot=0)
2. Create RoutingModel(manager)
3. Register time transit callback → time_matrix_seconds[from][to] + service_time
4. Add Time dimension (per-vehicle shift windows, overtime penalties)
5. Add Capacity dimension (per-vehicle weight limits, overcapacity penalties)
6. Set fixed vehicle dispatch costs
7. Add disjunction drop penalties
8. Register per-vehicle cost callbacks (distance × cost_per_km)
9. Configure search: SAVINGS + GUIDED_LOCAL_SEARCH (10s limit)
10. Solve → Decode solution into per-vehicle stop sequences
```

In [ ]:
# Stage 4 Code Reference: OR-Tools Solver Configuration
# This demonstrates the constraint setup (core logic from ortools_solver.py)

print("="*70)
print("OR-TOOLS CVRPTW SOLVER CONFIGURATION")
print("="*70)

solver_config = {
    "First Solution Strategy": "SAVINGS (Clarke-Wright savings heuristic)",
    "Metaheuristic": "GUIDED_LOCAL_SEARCH (penalty-augmented optimization)",
    "Time Limit": "10 seconds (configurable)",
    "Waiting Slack": "120 minutes (max wait at stop before window opens)",
}

for k, v in solver_config.items():
    print(f"  {k}: {v}")

print("\n" + "="*70)
print("CONSTRAINT CLASSES")
print("="*70)

constraints = [
    ("VehicleCapacityConstraint", "Max payload per vehicle", "Soft: 1,000,000 pts/kg over"),
    ("TimeWindowConstraint", "Customer delivery windows", "Soft: 100,000 pts/min late"),
    ("DriverShiftConstraint", "Driver work hours", "Soft: 50,000 pts/min overtime"),
    ("DropDisjunctionConstraint", "Zero-drop guarantee", "Hard: 1,000,000,000 pts/drop"),
    ("FleetActivationConstraint", "Vehicle dispatch cost", "Fixed: ₹2,000 per vehicle"),
]

print(f"{'Class':<30} {'Purpose':<30} {'Penalty'}")
print("-"*90)
for cls, purpose, penalty in constraints:
    print(f"{cls:<30} {purpose:<30} {penalty}")

### Penalty Point System — How the Solver Prioritizes

OR-Tools minimizes a **single scalar objective** that combines all costs:

```
Objective = Σ(travel_cost) + Σ(late_penalty) + Σ(overtime_penalty)
          + Σ(overcapacity_penalty) + Σ(dispatch_cost) + Σ(drop_penalty)
```

The **relative magnitudes** of penalties create a strict priority hierarchy:

| Priority | Constraint | Penalty Scale | Effect |
|----------|-----------|---------------|--------|
| 1 (Highest) | Don't drop orders | 1,000,000,000 | Near-zero drops guaranteed |
| 2 | Don't overload | 1,000,000/kg | Prevents vehicle overloading |
| 3 | Don't be late | 100,000/min | Strongly discourages late deliveries |
| 4 | Don't use overtime | 50,000/min | Discourages exceeding driver shift |
| 5 | Minimize dispatch | ₹2,000/vehicle | Minimizes fleet size |
| 6 (Lowest) | Minimize distance | ~₹8/km | Primary distance optimization |

**Key Insight:** The solver will NEVER drop an order (1B penalty) to save travel cost (~₹8/km). It will assign the order even if it means slight overtime (50K/min) or late delivery (100K/min). This is by design — the penalty magnitudes encode our business priorities.

In [ ]:
# Penalty magnitude comparison visualization

penalties = {
    "Drop Order": 1_000_000_000,
    "Overcapacity\n(per kg)": 1_000_000,
    "Late Delivery\n(per min)": 100_000,
    "Driver Overtime\n(per min)": 50_000,
    "Vehicle Dispatch\n(flat)": 2_000,
    "Travel Cost\n(per km)": 8,
}

print("Penalty Magnitude Comparison (log scale):")
print("="*60)
import math
max_log = math.log10(max(penalties.values()))

for name, value in penalties.items():
    name_clean = name.replace('\n', ' ')
    log_val = math.log10(max(value, 1))
    bar_len = int((log_val / max_log) * 40)
    bar = '█' * bar_len
    print(f"  {name_clean:<25} {value:>15,}  {bar}")

### Stage 5: Valhalla Route Geometry (ROUTE_GENERATION)

**Engine:** Valhalla `/route` API  
**Input:** Ordered waypoints per vehicle (depot → stop1 → stop2 → ... → depot)  

After OR-Tools determines the optimal stop sequence, Valhalla generates the **road-accurate route geometry** for each vehicle:

**API Call:**
```json
POST http://localhost:8002/route
{
    "locations": [
        {"lat": 19.055, "lon": 72.875, "type": "break"},
        {"lat": 19.136, "lon": 72.829, "type": "break"},
        {"lat": 19.017, "lon": 72.856, "type": "break"},
        {"lat": 19.055, "lon": 72.875, "type": "break"}
    ],
    "costing": "auto",
    "directions_options": {"units": "kilometers", "format": "geojson"}
}
```

**Output per vehicle:**
- GeoJSON `LineString` geometry (road-accurate polyline for Leaflet rendering)
- Turn-by-turn maneuver instructions ("Turn left on SV Road", "Take exit to WEH")
- Accurate road distance (km) and duration (minutes)

**Polyline6 Decoding:** Valhalla returns shapes as Polyline6 encoded strings (precision 1e-6). The client includes a custom decoder that converts to `[lon, lat]` arrays.

In [ ]:
# Stage 5 Code Reference: Valhalla Route Response Structure

import json

# Example Valhalla /route response (simplified)
valhalla_route_response = {
    "distance_km": 34.7,
    "duration_minutes": 62.3,
    "geometry": {
        "type": "LineString",
        "coordinates": [
            [72.8750, 19.0550],  # Depot
            [72.8712, 19.0623],  # Road segment
            [72.8645, 19.0801],  # Road segment
            [72.8434, 19.1124],  # Road segment
            [72.8296, 19.1364],  # Order 1 (Andheri)
            [72.8456, 19.0912],  # Road segment
            [72.8562, 19.0176],  # Order 2 (Dadar)
            [72.8750, 19.0550],  # Depot return
        ]
    },
    "maneuvers": [
        {"instruction": "Drive north on LBS Marg", "distance_km": 2.1, "duration_minutes": 5.2},
        {"instruction": "Turn left onto SV Road", "distance_km": 3.4, "duration_minutes": 8.1},
        {"instruction": "Continue to Andheri West", "distance_km": 5.8, "duration_minutes": 12.5},
    ],
    "source": "valhalla"
}

print("Valhalla /route Response:")
print(json.dumps(valhalla_route_response, indent=2))

---
## 5. Data Models

The system uses 4 SQLAlchemy models persisted to SQLite:

```
┌──────────────────┐       ┌───────────────┐       ┌──────────────┐
│ OptimizationJob  │ 1───* │    Order      │       │ VehicleFleet │
│                  │       │               │       │              │
│ id (UUID PK)     │       │ id (UUID PK)  │       │ id (INT PK)  │
│ filename         │       │ job_id (FK)   │       │ name         │
│ status           │       │ external_id   │       │ capacity_kg  │
│ current_step     │       │ raw_address   │       │ cost_per_km  │
│ progress_pct     │       │ latitude      │       │ shift_start  │
│ error_message    │       │ longitude     │       │ shift_end    │
│ total_orders     │       │ weight_kg     │       │ is_active    │
│ assigned_orders  │       │ service_time  │       └──────────────┘
│ vehicles_used    │       │ window_start  │
│ total_distance   │       │ window_end    │
│ total_cost       │       │ is_assigned   │
└────────┬─────────┘       │ vehicle_id    │
         │                 │ stop_sequence │
         │ 1───*           │ arrival_time  │
┌────────▼─────────┐       └───────────────┘
│     Route        │
│                  │
│ id (UUID PK)     │
│ job_id (FK)      │
│ vehicle_id       │
│ geometry (JSON)  │  ← GeoJSON LineString
│ maneuvers (JSON) │  ← Turn-by-turn
│ stop_ids (JSON)  │  ← Ordered UUIDs
└──────────────────┘
```

---
## 6. API Reference

### Jobs API

| Method | Endpoint | Description | Response |
|--------|----------|-------------|----------|
| `POST` | `/api/v1/jobs/upload` | Upload manifest, start optimization | `{job_id, status}` |
| `GET` | `/api/v1/jobs/{id}/status` | Lightweight poll (2s interval) | `{status, progress_pct, current_step}` |
| `GET` | `/api/v1/jobs/{id}` | Full detail with routes + geometry | `{routes[], orders[], summary}` |
| `GET` | `/api/v1/jobs` | List recent jobs | `[{job_id, status, ...}]` |
| `GET` | `/api/v1/jobs/{id}/export` | Download Excel report | `.xlsx` file |

### Fleet API

| Method | Endpoint | Description |
|--------|----------|-------------|
| `GET` | `/api/v1/fleet` | List all vehicles |
| `POST` | `/api/v1/fleet` | Add vehicle |
| `PUT` | `/api/v1/fleet/{id}` | Update vehicle |
| `DELETE` | `/api/v1/fleet/{id}` | Remove vehicle |

### Health API

| Method | Endpoint | Description |
|--------|----------|-------------|
| `GET` | `/api/v1/health` | DB + Valhalla connectivity check |

---
## 7. Valhalla Integration — Deep Dive

### 7.1 Valhalla APIs Used

| Endpoint | Purpose | Latency |
|----------|---------|----------|
| `POST /sources_to_targets` | N×N road distance + time matrix | ~50-200ms (20 locations) |
| `POST /route` | Road-accurate polyline + maneuvers | ~30-100ms per route |

### 7.2 Costing Models

| Profile | Best For | Considers |
|---------|----------|----------|
| `auto` | General delivery | Road hierarchy, turn costs |
| `truck` | Large freight | Height/weight restrictions |
| `motorcycle` | Two-wheeler | Lane splitting, narrow roads |

### 7.3 Fallback Strategy

If Valhalla is unreachable, the system degrades gracefully:
- **Matrix:** Haversine distance × 1.3 road detour factor, 30 km/h avg speed
- **Route:** Straight-line GeoJSON between waypoints

### 7.4 Valhalla vs PostgreSQL/pgRouting

| Metric | pgRouting (Previous) | Valhalla (Current) |
|--------|---------------------|---------------------|
| Matrix Latency (20 stops) | 3–8 seconds | 50–200 ms |
| Road Data | 843K segments in PostgreSQL | Pre-tiled OSM extract |
| Setup Complexity | PostgreSQL + PostGIS + pgRouting | Single Docker container |
| Turn-by-Turn | Not available | Full maneuver instructions |
| Traffic Support | Manual cost multipliers | Built-in costing models |

---
## 8. OR-Tools Integration — Deep Dive

### 8.1 Why OR-Tools and Not Just Valhalla?

Valhalla's `/optimized_route` endpoint solves only the **Traveling Salesman Problem (TSP)** for a **single vehicle** with **no constraints**. It answers: *"What is the shortest route visiting all stops?"*

Our problem requires:
- **Multi-vehicle assignment** (5+ vans, each with different capacities)
- **Capacity constraints** (each van has a weight limit)
- **Time window constraints** (each customer has a delivery window)
- **Driver shift enforcement** (each driver has work hours)
- **Penalty-based soft constraints** (overtime, late delivery, order dropping)

This is the **CVRPTW** — a fundamentally different (and much harder) problem that requires a dedicated optimization solver.

### 8.2 Solver Internal Flow

```
1. RoutingIndexManager  → Maps logical nodes to solver indices
       ↓
2. RoutingModel          → The optimization model container
       ↓
3. Transit Callbacks     → time_callback(i,j) = travel_time + service_time
       ↓                    cost_callback(i,j) = distance × cost_per_km
4. Dimensions            → Time dimension: per-vehicle shift windows
       ↓                    Capacity dimension: per-vehicle weight limits
5. Soft Upper Bounds     → Time windows: penalty/min late
       ↓                    Capacity: penalty/kg over
6. Disjunctions          → Drop penalty per unassigned node
       ↓
7. Search Parameters     → SAVINGS → GUIDED_LOCAL_SEARCH (10s)
       ↓
8. SolveWithParameters() → Returns feasible solution
       ↓
9. Solution Decoding     → Extract per-vehicle: stop_ids, arrival_times,
                           total_distance, total_duration, total_cost
```

### 8.3 Key OR-Tools Concepts

| Concept | Description | In Our Code |
|---------|-------------|-------------|
| **Dimension** | A cumulative quantity tracked along each route | `Time`, `Capacity` |
| **Transit Callback** | Function returning cost of traversing an arc | `time_matrix[from][to]` |
| **Soft Upper Bound** | Penalty for exceeding a limit (not hard-blocked) | `SetCumulVarSoftUpperBound()` |
| **Disjunction** | Allows dropping a node with penalty | `AddDisjunction()` |
| **Fixed Cost** | Per-vehicle activation cost | `SetFixedCostOfVehicle()` |
| **Finalizer** | Minimizes slack in dimension variables | `AddVariableMinimizedByFinalizer()` |

In [ ]:
# OR-Tools Solver — Pseudocode (from ortools_solver.py)

solver_pseudocode = """
╔══════════════════════════════════════════════════════════════════════╗
║            OR-Tools CVRPTW Solver — Key Steps                      ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  1. manager = RoutingIndexManager(                                 ║
║       num_nodes = N+1,          # depot + N orders                 ║
║       num_vehicles = M,         # active fleet size                ║
║       depot = 0                 # index 0 is depot                 ║
║     )                                                              ║
║                                                                    ║
║  2. routing = RoutingModel(manager)                                ║
║                                                                    ║
║  3. TIME CALLBACK:                                                 ║
║     def time_cb(from_idx, to_idx):                                 ║
║       return time_matrix[from_node][to_node] + service_time[to]    ║
║                                                                    ║
║  4. TIME DIMENSION:                                                ║
║     routing.AddDimension('Time', slack=120min, horizon=1440min)    ║
║     For each vehicle v:                                            ║
║       time_dim.CumulVar(start[v]).SetRange(shift_start, shift_end) ║
║       time_dim.SetSoftUpperBound(end[v], shift_end, 50K/min)      ║
║                                                                    ║
║  5. TIME WINDOWS (per customer):                                   ║
║     For each order i:                                              ║
║       time_dim.CumulVar(i).SetRange(window_start, horizon)        ║
║       time_dim.SetSoftUpperBound(i, window_end, 100K/min)         ║
║                                                                    ║
║  6. CAPACITY DIMENSION:                                            ║
║     routing.AddDimensionWithVehicleCapacity(demand_cb, caps)       ║
║     For each vehicle v:                                            ║
║       cap_dim.SetSoftUpperBound(end[v], capacity_kg, 1M/kg)       ║
║                                                                    ║
║  7. DISJUNCTIONS:                                                  ║
║     For each order: AddDisjunction([node_i], penalty=1B)           ║
║                                                                    ║
║  8. COST CALLBACKS (per vehicle):                                  ║
║     cost = distance_meters × cost_per_km / 1000                    ║
║                                                                    ║
║  9. SOLVE:                                                         ║
║     strategy = SAVINGS + GUIDED_LOCAL_SEARCH (10s)                 ║
║     solution = routing.SolveWithParameters(search_params)          ║
║                                                                    ║
║ 10. DECODE:                                                        ║
║     For each vehicle: extract stop_ids, arrival_times, metrics     ║
║                                                                    ║
╚══════════════════════════════════════════════════════════════════════╝
"""
print(solver_pseudocode)

---
## 9. Frontend Dashboard Architecture

### 9.1 Component Tree

```
App.tsx (Root State Manager)
│
├── Navbar.tsx                  ← Actions: Upload, Fleet, Export
├── UploadModal.tsx             ← Drag-and-drop file upload
├── FleetPanel.tsx              ← Vehicle CRUD modal
├── JobStatusTimeline.tsx       ← Pipeline progress (5 stages)
├── SummaryStats.tsx            ← 7 metric cards
│
├── RouteListSidebar.tsx        ← Left panel: route selection
├── RouteMap.tsx                ← Center: Leaflet map
│   ├── Dark CARTO tiles
│   ├── 12-color polylines from GeoJSON
│   ├── Numbered stop markers (red border if late)
│   └── Selection glow effects
│
├── RouteDetailPanel.tsx        ← Right panel: stop sequence
└── EmptyState.tsx              ← Welcome/landing screen
```

### 9.2 State Management

- **No external state library** — React `useState` + `useCallback` + `useMemo`
- **`useJobPolling` hook** — 2s interval polling, auto-fetches full payload on COMPLETED
- **`useFleet` hook** — Fleet CRUD lifecycle

### 9.3 Map Features

| Feature | Implementation |
|---------|----------------|
| Dark tiles | CARTO Dark Matter base layer |
| Route polylines | 12 distinct colors, rendered from GeoJSON |
| Stop markers | Numbered `DivIcon` with sequence number |
| Late detection | Red border + ⚠ badge on late stops |
| Route selection | Click polyline or sidebar → glow effect |
| Auto-fit | `fitBounds()` to show all routes |

---
## 10. Project Structure

```
logistics-route-optimization/
│
├── backend/
│   ├── app/
│   │   ├── main.py                    # FastAPI entry point
│   │   ├── config.py                  # Pydantic Settings (env-driven)
│   │   ├── api/                       # REST endpoints (jobs, fleet, health)
│   │   ├── models/entities.py         # SQLAlchemy models
│   │   ├── schemas/                   # Pydantic response schemas
│   │   ├── services/
│   │   │   ├── pipeline.py            # PipelineOrchestrator
│   │   │   ├── excel_parser.py        # Manifest ingestion
│   │   │   ├── geocoding/             # Pluggable geocoders
│   │   │   ├── valhalla/client.py     # Valhalla API client
│   │   │   └── optimization/          # OR-Tools solver + constraints
│   │   ├── database/session.py        # Async DB engine
│   │   └── utils/                     # Logger, time utils
│   ├── tests/                         # Unit tests
│   └── requirements.txt               # Python deps
│
├── frontend/
│   ├── src/
│   │   ├── App.tsx                    # Root component
│   │   ├── components/               # 9 UI components
│   │   ├── hooks/                    # Custom React hooks
│   │   ├── services/api.ts           # Axios client
│   │   └── types/index.ts            # TypeScript types
│   └── package.json
│
├── data/
│   ├── sample_orders.csv
│   └── sample_orders.xlsx
│
├── README.md
└── ARCHITECTURE.md
```

---
## 11. Setup & Running

### Prerequisites
- Python 3.10+
- Node.js 18+
- Docker (for Valhalla)

### Backend
```bash
cd backend
python -m venv venv
venv\Scripts\activate
pip install -r requirements.txt
uvicorn app.main:app --reload --port 8000
```

### Frontend
```bash
cd frontend
npm install
npm run dev   # → http://localhost:5173
```

### Valhalla (Docker)
```bash
docker run -dt --name valhalla \
  -p 8002:8002 \
  -v valhalla_data:/data \
  -e tile_urls=https://download.geofabrik.de/asia/india/maharashtra-latest.osm.pbf \
  ghcr.io/gis-ops/valhalla:latest
```

---
## Summary

### Key Findings

- **Valhalla** handles all road-network operations: distance/time matrix computation (`/sources_to_targets`) and road-accurate route geometry generation (`/route`). It replaces the previous PostgreSQL + pgRouting setup with **20-40× faster** matrix computation.

- **Google OR-Tools** handles the fleet optimization: it takes Valhalla's distance matrix and solves the multi-vehicle CVRPTW problem with 5 constraint classes (capacity, time windows, driver shifts, dispatch costs, drop penalties).

- The **penalty point system** creates a strict priority hierarchy ensuring the solver never drops an order to save travel cost, and never overloads a vehicle to avoid a late delivery.

- The **pipeline architecture** (6 sequential stages) enables real-time progress tracking on the frontend via 2-second polling.

### Insights & Next Steps

- **Valhalla's `/optimized_route`** (TSP solver) could potentially replace OR-Tools for single-vehicle scenarios, but it lacks multi-vehicle support, capacity constraints, and time window enforcement — making OR-Tools essential for the fleet optimization use case.

- **Future enhancements:** Real-time traffic integration via Valhalla costing options, avoid-zone polygons for restricted areas, and Valhalla `truck` costing for heavy vehicle routing with height/weight restrictions.